# Mobilis Telecom - Intent Classification (DistilBERT) — v2

**What changed vs. the "fixed" version:**

The leakage-safe split from the previous version is kept as-is (it was correct). What's new here fixes the
*next* problem that showed up once real-world sentences were tested against it: 4 missing intents and several
under-represented intents were patched in with ~20 hand-written examples each, but 20 examples against
hundreds of templated rows per intent is not enough signal for the model to actually learn the new phrasing —
it mostly still learned "what a templated Bitext sentence looks like." That's why `"My internet is very slow"`
was routed to `activate_roaming` at 47% confidence instead of `report_poor_signal_coverage`.

**Fixes in this version, in order of impact:**
1. **Oversampling** — new/real-world examples are replicated so they're not a rounding error next to hundreds
   of templated rows.
2. **Downsampling the templated majority** — each original intent is capped so a few hundred near-duplicate
   templates can't drown out everything else.
3. **Class-weighted loss** — a custom `Trainer` weights the loss inversely to class frequency, which is more
   principled than oversampling alone and doesn't bloat the dataset.
4. **A genuinely held-out real-world evaluation set** — new hand-written sentences, never seen during training
   or augmentation, used both to report honest accuracy and to tune the confidence threshold. Tuning the
   threshold against the templated test split (like the previous version did) gives an optimistic threshold
   that doesn't transfer to real user phrasing.
5. **New/real-world rows are excluded from template clustering** — since they're now deliberately duplicated,
   clustering would lump all copies into one cluster and could push all of them to one side of the split.
   Each is instead treated as its own cluster so they distribute normally across train/val/test.


In [ ]:
!nvidia-smi

## 1. Install Required Libraries

In [ ]:
!pip install -q transformers datasets accelerate evaluate

## 2. Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from collections import defaultdict
from difflib import SequenceMatcher
from scipy.special import softmax
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)
from datasets import Dataset
import evaluate
import warnings
warnings.filterwarnings("ignore")


## 3. Load the Base Dataset

In [ ]:
df = pd.read_csv("hf://datasets/bitext/Bitext-telco-llm-chatbot-training-dataset/bitext-telco-llm-chatbot-training-dataset.csv")
df["tag"] = "TEMPLATED"
print(df.shape)
df.head()


## 4. Add Missing Intents

The original label set has no intent at all for password/account recovery, store locations, general technical
support, or "explain my bill" — real user needs that were simply never labeled. These have to be added as
real intents with real training examples; a confidence-threshold fallback can't invent a category that was
never in the data.

In [ ]:
NEW_INTENTS = {
    "account_recovery": [
        "I forgot my password, can you help me reset it?",
        "How do I recover my account password?",
        "I can't log in, I need to reset my password",
        "My password isn't working, I need to reset it",
        "I forgot my login credentials, please help",
        "How can I change my password?",
        "I need to recover my account, I forgot my password",
        "Password reset help needed",
        "I can't access my account, forgot password",
        "Reset my password please",
        "How do I reset my Mobilis account password?",
        "I forgot my Mobilis password, what should I do?",
        "Can you help me recover my account?",
        "My password is not working, need to reset",
        "I need a new password for my account",
    ],
    "store_location": [
        "Where is the nearest Mobilis store?",
        "Find a Mobilis store near me",
        "What are your store locations?",
        "Nearest Mobilis shop location",
        "Where can I find a Mobilis store?",
        "Mobilis store near my location",
        "Store locator for Mobilis",
        "Where is the closest Mobilis branch?",
        "Mobilis retail store locations",
        "Find a store in my area",
        "What are the store hours for Mobilis?",
        "Nearest Mobilis service center",
        "Where is the Mobilis office located?",
        "Mobilis shop address",
        "Mobilis branches in my city",
    ],
    "technical_support": [
        "My SIM card stopped working",
        "My phone isn't connecting to the network",
        "I can't make calls, what's wrong?",
        "My device is not working properly",
        "Technical issue with my mobile service",
        "My phone has no signal",
        "I need technical help with my device",
        "Phone not connecting to network",
        "Technical support needed for my phone",
        "My SIM card is not being recognized",
        "I have a technical problem with my service",
        "My phone is showing no service",
        "My device has a problem, need assistance",
        "My phone keeps disconnecting",
        "Need help with my SIM card issue",
    ],
    "bill_explanation": [
        "Why is my bill higher than usual?",
        "Explain my phone bill charges",
        "I don't understand my bill",
        "Can you explain these charges on my bill?",
        "My bill has unexpected charges",
        "Why did my bill increase this month?",
        "I need help understanding my bill",
        "Bill clarification needed",
        "What are these charges on my account?",
        "My bill is confusing, please explain",
        "Why is my bill so expensive this month?",
        "Help me understand my monthly bill",
        "What do these charges mean on my bill?",
        "Can you break down my bill for me?",
        "Why did my charges go up?",
    ],
}

new_rows = []
for intent, examples in NEW_INTENTS.items():
    for example in examples:
        new_rows.append({
            "instruction": example,
            "intent": intent,
            "category": "SUPPORT",
            "tags": "NEW",
            "tag": "NEW",
            "response": f"Thank you for contacting Mobilis support regarding your {intent.replace('_', ' ')} request. A representative will assist you shortly.",
        })
new_df = pd.DataFrame(new_rows)
print(f"{len(new_rows)} new-intent examples across {len(NEW_INTENTS)} new intents")


## 5. Add Real-World Phrasing to Existing Under-Represented Intents

In [ ]:
REAL_WORLD_EXAMPLES = {
    "report_poor_signal_coverage": [
        "My internet is very slow today",
        "The internet keeps disconnecting",
        "I have no internet connection in my area",
        "My WiFi is down, I need help",
        "Internet connection is unstable",
        "My mobile data is not working well",
        "Slow internet speeds, what's wrong?",
        "I can't get a good signal",
        "Network coverage is poor here",
        "My internet is not working at all",
        "The connection keeps dropping",
        "No signal in my location",
        "I have terrible reception at home",
        "Internet is too slow to use",
        "Network is down in my area",
    ],
    "dispute_invoice": [
        "I don't recognize these charges on my bill",
        "There's a charge on my bill that shouldn't be there",
        "My bill has an error, please fix it",
        "I was overcharged on my last bill",
        "There are incorrect charges on my statement",
        "I'm being charged for something I didn't use",
        "Please review my bill, I think there's a mistake",
        "Why am I being charged for international calls?",
        "My bill is wrong, I need it corrected",
        "I didn't make these calls on my bill",
        "There's an unauthorized charge on my account",
        "I need to dispute a charge on my bill",
        "I was charged for a service I didn't use",
        "My bill shows roaming charges but I didn't roam",
        "There's a billing error on my statement",
    ],
    "invoices": [
        "I want to view my bill online",
        "Where can I see my invoice?",
        "Download my monthly bill",
        "I need a copy of my bill",
        "How to access my billing statement",
        "Where do I find my invoice?",
        "I want to see my current bill",
        "Get my bill summary",
        "View my latest invoice",
        "I need my billing history",
        "Show me my monthly charges",
        "Where can I download my invoice?",
        "Access my billing statement",
        "How to view my invoice online",
        "Where can I find my previous bills?",
    ],
    "change_plan": [
        "I want to switch to a cheaper plan",
        "How to upgrade my mobile plan",
        "I need to change my internet plan",
        "What plans are available for upgrade?",
        "Can I change to a different plan?",
        "I want to downgrade my plan",
        "How to change my data plan",
        "I need a different subscription",
        "Upgrade my mobile plan please",
        "I want a plan with more data",
        "I need to switch to a family plan",
        "What are my plan upgrade options?",
        "Can I get a better plan?",
        "I need a plan with more minutes",
        "I want to switch plans",
    ],
    "cancel_plan": [
        "I want to cancel my subscription",
        "How to terminate my plan?",
        "I need to end my contract",
        "Cancel my mobile service please",
        "I want to stop my subscription",
        "How to disconnect my service",
        "I need to cancel my internet plan",
        "Terminate my account please",
        "I want to end my phone contract",
        "How to cancel my mobile plan?",
        "I need to unsubscribe from the service",
        "I want to close my account",
        "I need to end my service with Mobilis",
        "Please cancel my plan",
        "How to disconnect my line",
    ],
    "customer_service": [
        "I need to speak to a real person",
        "Get me to a human agent",
        "I want to talk to customer service",
        "Can I talk to a representative?",
        "I need human assistance",
        "Speak to a customer service agent",
        "I want to talk to someone about my bill",
        "Connect me to a live agent",
        "I need help from a real person",
        "Customer service representative please",
        "I want to speak with a manager",
        "Transfer me to a human",
        "Can I speak to someone in person?",
        "Talk to a real person about my issue",
        "Get me a live agent",
    ],
}

real_rows = []
for intent, examples in REAL_WORLD_EXAMPLES.items():
    existing = df[df["intent"] == intent]
    category = existing.iloc[0]["category"] if len(existing) else "SUPPORT"
    response_template = existing.iloc[0]["response"] if len(existing) else f"Thank you for contacting Mobilis regarding your {intent.replace('_', ' ')} request."
    for example in examples:
        real_rows.append({
            "instruction": example,
            "intent": intent,
            "category": category,
            "tags": "REAL",
            "tag": "REAL",
            "response": response_template,
        })
real_df = pd.DataFrame(real_rows)
print(f"{len(real_rows)} real-world examples added across {len(REAL_WORLD_EXAMPLES)} existing intents")


## 6. Oversample the New/Real-World Rows

~15 examples per intent is roughly 2–4% of that intent's templated row count. At that ratio, standard
cross-entropy training barely moves the decision boundary — the gradient signal from the minority phrasing
gets swamped by the majority templated phrasing every batch. Replicating each new/real row several times
brings their effective count close to parity with a capped templated intent (set up in the next step), so the
model actually has to fit them, not just tolerate them.

In [ ]:
OVERSAMPLE_FACTOR = 8  # ~15 examples * 8 = ~120, comparable to the templated cap below

new_df_os = pd.concat([new_df] * OVERSAMPLE_FACTOR, ignore_index=True)
real_df_os = pd.concat([real_df] * OVERSAMPLE_FACTOR, ignore_index=True)

print(f"new_df: {len(new_df)} -> {len(new_df_os)} after oversampling")
print(f"real_df: {len(real_df)} -> {len(real_df_os)} after oversampling")


## 7. Downsample the Templated Majority

The templated rows are heavily redundant — most of a given intent's few hundred rows differ by one swapped
word. Capping each *original* intent at a fixed number of rows removes redundancy without losing signal, and
stops the templated majority from drowning out the augmented intents even after oversampling.

In [ ]:
TEMPLATE_CAP = 150

def cap_group(g, n=TEMPLATE_CAP):
    return g.sample(n=min(len(g), n), random_state=42)

df_capped = df.groupby("intent", group_keys=False).apply(cap_group).reset_index(drop=True)
print(f"Templated rows: {len(df)} -> {len(df_capped)} after capping at {TEMPLATE_CAP}/intent")

df = pd.concat([df_capped, new_df_os, real_df_os], ignore_index=True)
print(f"\nFinal combined dataset: {len(df)} rows, {df['intent'].nunique()} intents")
print(df["tag"].value_counts())


## 8. Explore the Rebalanced Dataset

In [ ]:
plt.figure(figsize=(10, 9))
sns.countplot(y=df["intent"], order=df["intent"].value_counts().index,
              hue=df["tag"], dodge=False)
plt.title("Examples per intent after capping + oversampling")
plt.tight_layout()
plt.show()

df["intent"].value_counts()


## 9. Keyword Lookup Helper
Use this **before** assuming a phrase belongs to a given intent — shows which intents actually contain a
keyword in their training examples, and how many examples support it.

In [ ]:
def inspect_keyword(keyword, n=15):
    matches = df[df["instruction"].str.contains(keyword, case=False, na=False)]
    print(f'"{keyword}" appears in {len(matches)} training examples\n')
    print("Distribution across intents:")
    print(matches["intent"].value_counts())
    print()
    display(matches[["instruction", "intent", "tag"]].head(n))
    return matches

_ = inspect_keyword("slow")


## 10. Data Cleaning & Label Encoding

In [ ]:
df = df.drop_duplicates(subset=["instruction", "intent"]).dropna(subset=["instruction", "intent"]).reset_index(drop=True)
df["instruction"] = df["instruction"].str.lower().str.strip()

labels = sorted(df.intent.unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label"] = df["intent"].map(label2id)
print(f"{len(labels)} intents:", labels)


## 11. Template Clustering (Leakage Fix)

New/real rows are now deliberate duplicates of each other, so clustering them would collapse every oversampled
copy of a sentence into one cluster and risk pushing all copies to the same side of the split. Each new/real
row is instead given its own singleton cluster; only the original templated rows go through similarity
clustering.

In [ ]:
def build_template_clusters(df, text_col="instruction", intent_col="intent",
                             threshold=0.85, prefix_words=4):
    n = len(df)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    buckets = defaultdict(list)
    for i, row in df.iterrows():
        prefix = " ".join(row[text_col].split()[:prefix_words])
        buckets[(row[intent_col], prefix)].append(i)

    for idxs in buckets.values():
        if len(idxs) < 2:
            continue
        texts = df.loc[idxs, text_col].tolist()
        for a in range(len(idxs)):
            for b in range(a + 1, len(idxs)):
                if SequenceMatcher(None, texts[a], texts[b]).ratio() > threshold:
                    union(idxs[a], idxs[b])

    return np.array([find(i) for i in range(n)])

templated_mask = df["tag"] == "TEMPLATED"

df["cluster"] = -1
templated_clusters = build_template_clusters(df[templated_mask].reset_index(drop=True))
df.loc[templated_mask, "cluster"] = templated_clusters

# Every NEW/REAL row gets its own singleton cluster so it distributes normally across the split
next_id = df["cluster"].max() + 1
n_singleton = (~templated_mask).sum()
df.loc[~templated_mask, "cluster"] = np.arange(next_id, next_id + n_singleton)

n_clusters = df["cluster"].nunique()
print(f"{len(df)} rows -> {n_clusters} clusters ({templated_mask.sum()} templated rows clustered, "
      f"{n_singleton} new/real rows kept singleton)")


## 12. Leakage-Safe Train/Validation/Test Split

In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df["cluster"]))

temp_df = df.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx_rel, test_idx_rel = next(gss2.split(temp_df, groups=temp_df["cluster"]))
val_idx = temp_df.iloc[val_idx_rel].index.to_numpy()
test_idx = temp_df.iloc[test_idx_rel].index.to_numpy()

train_df = df.loc[train_idx].reset_index(drop=True)
val_df = df.loc[val_idx].reset_index(drop=True)
test_df = df.loc[test_idx].reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
print("\nOverlap check (should all be 0):")
print("train/val:", len(set(train_df['cluster']) & set(val_df['cluster'])))
print("train/test:", len(set(train_df['cluster']) & set(test_df['cluster'])))
print("val/test:", len(set(val_df['cluster']) & set(test_df['cluster'])))

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_enc = tokenizer(list(train_df["instruction"]), truncation=True, padding=True, max_length=64)
val_enc = tokenizer(list(val_df["instruction"]), truncation=True, padding=True, max_length=64)
test_enc = tokenizer(list(test_df["instruction"]), truncation=True, padding=True, max_length=64)

train_ds = Dataset.from_dict(train_enc).add_column("labels", list(train_df["label"]))
val_ds = Dataset.from_dict(val_enc).add_column("labels", list(val_df["label"]))
test_ds = Dataset.from_dict(test_enc).add_column("labels", list(test_df["label"]))

train_texts, val_texts, test_texts = train_df["instruction"], val_df["instruction"], test_df["instruction"]
train_labels, val_labels, test_labels = train_df["label"], val_df["label"], test_df["label"]


## 13. Confirm the Leakage Is Actually Gone

In [ ]:
def near_duplicate_rate(train_texts, test_texts, sample_train=800, sample_test=80, threshold=0.9, seed=1):
    tr_sample = pd.Series(train_texts).sample(min(sample_train, len(train_texts)), random_state=seed).tolist()
    te_sample = pd.Series(test_texts).sample(min(sample_test, len(test_texts)), random_state=seed).tolist()
    near_dupes = 0
    for t in te_sample:
        if any(SequenceMatcher(None, t, tr).ratio() > threshold for tr in tr_sample):
            near_dupes += 1
    rate = near_dupes / len(te_sample)
    print(f"{near_dupes}/{len(te_sample)} sampled test examples ({rate:.0%}) are near-duplicates of a train example")
    return rate

_ = near_duplicate_rate(train_texts, test_texts)


## 14. Class-Weighted Loss

Oversampling helps, but a custom `Trainer` with a class-weighted `CrossEntropyLoss` (weights inversely
proportional to each class's frequency in `train_df`) is the more principled fix — it corrects the imbalance
directly in the loss rather than relying purely on duplicated rows, and it also helps the templated intents
that ended up smaller after capping.

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(labels)),
    y=train_df["label"].values,
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float)
print("Class weights (min/max):", class_weights_t.min().item(), class_weights_t.max().item())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_ = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_t.to(logits.device))
        loss = loss_fct(logits.view(-1, len(labels)), labels_.view(-1))
        return (loss, outputs) if return_outputs else loss


## 15. Build and Train DistilBERT

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(labels), id2label=id2label, label2id=label2id
)

acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels_ = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc.compute(predictions=preds, references=labels_)

args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()


## 16. Evaluate on the Leakage-Safe Test Split
This is still templated-style data (same distribution as train, minus leakage) — useful as a sanity check, but
not the real signal for production behavior. That comes in step 19.

In [ ]:
predictions_test = trainer.predict(test_ds)
preds = np.argmax(predictions_test.predictions, axis=1)

label_ids = list(range(len(labels)))  # 0..29, matches id2label/label2id

print(classification_report(
    test_labels, preds,
    labels=label_ids,
    target_names=labels,
    zero_division=0
))

cm = confusion_matrix(test_labels, preds, labels=label_ids)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, xticklabels=labels, yticklabels=labels, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 17. Inspect Misclassified Examples

In [ ]:
test_texts_reset = test_texts.reset_index(drop=True)
test_labels_reset = test_labels.reset_index(drop=True)

wrong_idx = np.where(preds != test_labels_reset.values)[0]
print(f"{len(wrong_idx)} / {len(test_labels_reset)} test examples misclassified\n")

for i in wrong_idx[:20]:
    print(f"TEXT: {test_texts_reset[i]}")
    print(f"TRUE: {labels[test_labels_reset[i]]}  |  PRED: {labels[preds[i]]}")
    print("-" * 60)


## 18. Save the Model

In [ ]:
trainer.save_model("mobilis_intent_model")
tokenizer.save_pretrained("mobilis_intent_model")
print("Model saved to 'mobilis_intent_model'")

## 19. Held-Out Real-World Evaluation Set

These sentences were never used for training or augmentation — different wording from `NEW_INTENTS` /
`REAL_WORLD_EXAMPLES` above, even for the same intents. This is the honest signal for production behavior,
and the set the confidence threshold gets tuned against below, instead of against the templated test split.

In [ ]:
REAL_WORLD_EVAL = [
    ("I want to install internet at my home.", "install_internet"),
    ("My internet connection is not working.", "report_poor_signal_coverage"),
    ("My internet is very slow.", "report_poor_signal_coverage"),
    ("The wifi keeps cutting out on me.", "report_poor_signal_coverage"),
    ("I forgot my password.", "account_recovery"),
    ("Can't log into my account, locked out.", "account_recovery"),
    ("I want to cancel my internet subscription.", "cancel_plan"),
    ("Please shut down my line, I don't need it anymore.", "cancel_plan"),
    ("I don't recognize a charge on my bill.", "dispute_invoice"),
    ("This charge looks wrong, can you check it?", "dispute_invoice"),
    ("Why is my bill higher than usual?", "bill_explanation"),
    ("Not sure what this fee on my statement is for.", "bill_explanation"),
    ("I want to know how much data I have used.", "check_excess_data_charges"),
    ("I want to change my mobile plan.", "change_plan"),
    ("Can I move to a bigger data package?", "change_plan"),
    ("I need a technician to come to my house.", "technical_support"),
    ("Someone needs to come fix my line in person.", "technical_support"),
    ("My SIM card stopped working.", "technical_support"),
    ("Where is the nearest Mobilis store?", "store_location"),
    ("Is there a shop near downtown I can visit?", "store_location"),
    ("Get me a human, this bot isn't helping.", "customer_service"),
    ("I'd like to talk to an actual agent please.", "customer_service"),
    ("Can I see my last few invoices?", "invoices"),
    ("Send me a copy of this month's bill.", "invoices"),
]

# Filter out any label not present in the trained label set (keeps this cell safe to edit)
REAL_WORLD_EVAL = [(t, l) for t, l in REAL_WORLD_EVAL if l in labels]
print(f"{len(REAL_WORLD_EVAL)} held-out real-world evaluation examples across "
      f"{len(set(l for _, l in REAL_WORLD_EVAL))} intents")


## 20. Find the Confidence Threshold — Tuned on the Real-World Set

Tuning against the templated test split (as the previous version did) gives an optimistic threshold, since
templated test phrasing is close to templated train phrasing by construction. Tuning against the held-out
real-world set instead gives a threshold that reflects what will actually happen with real user messages.

In [ ]:
classifier = pipeline(
    "text-classification",
    model="./mobilis_intent_model_v2",
    tokenizer="./mobilis_intent_model_v2",
    top_k=5,
)

rw_texts = [t for t, _ in REAL_WORLD_EVAL]
rw_true = [l for _, l in REAL_WORLD_EVAL]
rw_results = classifier([t.lower() for t in rw_texts])
rw_top_labels = [r[0]["label"] for r in rw_results]
rw_top_scores = np.array([r[0]["score"] for r in rw_results])

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
rows = []
for thr in thresholds:
    final = [rw_top_labels[i] if rw_top_scores[i] >= thr else "unknown" for i in range(len(rw_true))]
    correct = sum(1 for i in range(len(rw_true)) if final[i] == rw_true[i])
    unknown_rate = sum(1 for f in final if f == "unknown") / len(final)
    rows.append({"threshold": thr, "accuracy": correct / len(rw_true), "unknown_rate": unknown_rate})

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

best = results_df.loc[results_df["accuracy"].idxmax()]
OPTIMAL_THRESHOLD = float(best["threshold"])
print(f"\nRecommended threshold (tuned on real-world set): {OPTIMAL_THRESHOLD}")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(results_df["threshold"], results_df["accuracy"], "b-o", label="Accuracy")
ax1.set_xlabel("Confidence Threshold")
ax1.set_ylabel("Accuracy", color="b")
ax2 = ax1.twinx()
ax2.plot(results_df["threshold"], results_df["unknown_rate"], "r-s", label="Unknown Rate")
ax2.set_ylabel("Unknown Rate", color="r")
plt.title("Threshold tuned on held-out real-world sentences")
plt.tight_layout()
plt.show()


## 21. Inference With Confidence-Threshold Fallback

In [ ]:
def classify_with_fallback(text, threshold=OPTIMAL_THRESHOLD):
    """Return (intent, confidence, top5). Falls back to 'unknown' below the threshold
    so the downstream RAG/LLM step never treats a shaky guess as certain."""
    results = classifier(text.lower())[0]
    top = results[0]
    if top["score"] < threshold:
        return "unknown", top["score"], results
    return top["label"], top["score"], results

print("=" * 70)
print("REAL-WORLD EVALUATION (held-out set, with true labels)")
print("=" * 70)
n_correct = 0
for text, true_label in REAL_WORLD_EVAL:
    intent, score, top5 = classify_with_fallback(text)
    ok = "OK " if intent == true_label else "ERR"
    n_correct += (intent == true_label)
    print(f"\n[{ok}] {text}")
    print(f"     true: {true_label}  |  routed as: {intent}  (top1 = {top5[0]['label']} @ {top5[0]['score']:.3f})")

print(f"\n{n_correct}/{len(REAL_WORLD_EVAL)} correctly routed ({n_correct/len(REAL_WORLD_EVAL):.0%})")


## 22. New-Intent / Out-of-Scope Detection

In [ ]:
def detect_new_intent(text, threshold=OPTIMAL_THRESHOLD):
    """Flag a query as likely belonging to an intent that isn't in the label set at all."""
    results = classifier(text.lower())[0]
    top_score = results[0]["score"]
    return {
        "is_new_intent": top_score < threshold,
        "confidence": top_score,
        "suggested_intent": "unknown" if top_score < threshold else results[0]["label"],
        "top_predictions": results,
    }

test_new_queries = [
    "I want to change my email address",
    "How do I update my contact information?",
    "My phone was stolen, what should I do?",
    "I need a new SIM card",
    "Can I port my number to another network?",
]

for query in test_new_queries:
    result = detect_new_intent(query)
    print(f"\nQuery: {query}")
    print(f"Is new intent? {result['is_new_intent']}  |  confidence: {result['confidence']:.3f}  "
          f"|  suggested: {result['suggested_intent']}")


## 23. Notes on Remaining Gaps

- **`TEMPLATE_CAP`, `OVERSAMPLE_FACTOR`, and threshold are all tunable** — if a specific intent still
  underperforms in step 19, raise its oversampling or lower the cap on nearby templated intents first before
  touching the confidence threshold.
- **The held-out real-world set is still small (~20 examples).** It's enough to catch the kind of failure seen
  earlier (a whole intent's phrasing being unlearned), but not enough to trust a precise accuracy number.
  Growing it with real user queries once this is deployed is the next step, and is more valuable than further
  synthetic augmentation.
- **Overlapping intents remain a genuine ambiguity, not a bug.** `technical_support`, `report_poor_signal_coverage`,
  and `account_recovery` all cover "my X isn't working" phrasing. Some misroutes between these three are
  expected and may need a business decision (e.g. merge intents, or route to `technical_support` as a shared
  default) rather than a modeling fix.
- **Retrain periodically on `unknown`-routed production queries** once real user data is flowing in — synthetic
  real-world examples are a stand-in for that, not a substitute.


## 24. (Optional) Save to Google Drive
Only relevant when running in Google Colab.

In [ ]:


from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copytree("mobilis_intent_model", "/content/drive/MyDrive/mobilis_intent_model", dirs_exist_ok=True)
print("Saved locally and copied to Drive")